# LMSYS Chatbot Arena — Training (Gemma2-2B QLoRA)

Fine-tune Gemma2-2B-IT with QLoRA for preference prediction.  
**Run on Kaggle with 2×T4 GPUs** (or any GPU with ≥16GB VRAM).

### Required Kaggle Datasets (add as Input):
- `lmsys-chatbot-arena` (competition data)
- `gemma-2-2b-it` 
- Optional: `lmsys-33k-deduplicated` 

In [ ]:
pip install -U bitsandbytes>=0.46.1

In [ ]:
# Force single GPU for 2B — Trainer sees all GPUs and multiplies per_device_batch_size by n_gpu,
# doubling the effective batch and halving training steps. Must be set before torch import.
# For 9B we need both GPUs via device_map='auto', so don't restrict.
import os as _os

# ============================================================
# CONFIG — Change these settings to tune performance
# ============================================================
class CFG:
    # Model size: '2b' or '9b'
    model_size = '2b'
    
    # Paths (auto-configured below based on model_size)
    data_path = '/kaggle/input/competitions/lmsys-chatbot-arena'
    output_dir = '/kaggle/working/lora-adapter'
    
    # External data (set to None to skip)
    external_data_path = None  # '/kaggle/input/lmsys-33k-deduplicated/lmsys-33k-deduplicated.csv'
    
    # LoRA config (auto-adjusted below for 9B)
    lora_r = 64
    lora_alpha = 128  # alpha/r = 2.0 — was 16 (scaling=0.25) which starved learning
    lora_dropout = 0.05
    lora_target_modules = [
        'q_proj', 'k_proj', 'v_proj',
        'o_proj', 'gate_proj', 'up_proj', 'down_proj'
    ]
    freeze_layers = 0  # number of layers to freeze (0 = train all)
    
    # Training config (auto-adjusted below for 9B)
    max_length = 512
    max_steps = 1500           # cap training — loss plateaus early, no need for full epoch
    learning_rate = 1e-4
    warmup_steps = 50
    weight_decay = 0.01
    label_smoothing = 0.0  # try 0.03 for Phase 4
    
    # Quantization
    use_4bit = True
    bnb_4bit_compute_dtype = 'float16'
    bnb_4bit_quant_type = 'nf4'
    use_double_quant = True
    
    # Misc
    seed = 42
    num_labels = 3

# Auto-configure model path AND batch settings based on model_size
if CFG.model_size == '2b':
    _os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # single GPU for 2B
    CFG.model_path = '/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2'
    # 2B model (~1.5GB in 4-bit) + grad_ckpt fits batch=16 on one T4 (16GB)
    CFG.per_device_batch_size = 16
    CFG.gradient_accumulation_steps = 1  # effective batch = 16 × 1 = 16
    CFG.device_map = {'': 0}  # single GPU — 'auto' breaks gradient flow with PEFT+grad_ckpt
else:
    # 9B needs both GPUs — do NOT restrict CUDA_VISIBLE_DEVICES
    CFG.model_path = '/kaggle/input/models/google/gemma-2/transformers/gemma-2-9b-it/2'
    # 9B model (~5GB in 4-bit) needs both T4s — tight on time, maximize throughput
    CFG.per_device_batch_size = 1
    CFG.gradient_accumulation_steps = 16  # effective batch = 16 (fewer optimizer steps)
    CFG.device_map = 'auto'  # split across GPUs
    # 9B-specific tuning
    CFG.learning_rate = 2e-5       # lower LR for larger model
    CFG.max_length = 768           # balance context vs speed (1024 too slow for time limit)
    CFG.max_steps = 1500           # same cap — 9B also plateaus early
    CFG.warmup_steps = 50          # fewer warmup since fewer total steps
    CFG.freeze_layers = 10         # freeze early layers to save VRAM
    CFG.lora_r = 32                # lower rank sufficient for 9B capacity
    CFG.lora_alpha = 64            # keep scaling = alpha/r = 2.0

# Validate paths exist
if not _os.path.exists(CFG.model_path):
    print(f'WARNING: Model path not found: {CFG.model_path}')
    print('On Kaggle: Add Input > Models > search "gemma-2" > select the correct variant')
if not _os.path.exists(CFG.data_path):
    print(f'WARNING: Data path not found: {CFG.data_path}')
    print('On Kaggle: Add Input > Competitions > search "lmsys-chatbot-arena"')

eff_batch = CFG.per_device_batch_size * CFG.gradient_accumulation_steps
print(f'Model: gemma-2-{CFG.model_size}-it | LoRA r={CFG.lora_r}, alpha={CFG.lora_alpha} (scaling={CFG.lora_alpha/CFG.lora_r})')
print(f'Batch: {CFG.per_device_batch_size} × {CFG.gradient_accumulation_steps} = {eff_batch} effective')
print(f'Device map: {CFG.device_map}')
print(f'LR: {CFG.learning_rate}, max_length: {CFG.max_length}, freeze_layers: {CFG.freeze_layers}')
print(f'Max steps: {CFG.max_steps}')


On Kaggle: Add Input > Models > search "gemma-2" > select the correct variant
Model: gemma-2-2b-it | LoRA r=64, alpha=128 (scaling=2.0)
Batch: 4 × 2 = 8 effective
Device map: {'': 0}
Estimated steps: ~7184


In [2]:
import os
import json
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType,
)
from sklearn.metrics import log_loss
import warnings
warnings.filterwarnings('ignore')

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)} — {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB')

PyTorch: 2.10.0+cu128
CUDA available: True
  GPU 0: Tesla T4 — 15.6 GB
  GPU 1: Tesla T4 — 15.6 GB


## 1. Load & Prepare Data

In [3]:
train_df = pd.read_csv(f'{CFG.data_path}/train.csv')
print(f'Competition train: {len(train_df)} rows')

# Add external data if configured
if CFG.external_data_path and os.path.exists(CFG.external_data_path):
    ext_df = pd.read_csv(CFG.external_data_path)
    # Ensure same columns
    required_cols = ['id', 'prompt', 'response_a', 'response_b',
                     'winner_model_a', 'winner_model_b', 'winner_tie']
    ext_df = ext_df[[c for c in required_cols if c in ext_df.columns]]
    train_df = pd.concat([ext_df, train_df], ignore_index=True)
    print(f'+ External data: total {len(train_df)} rows')

# Use 100% of data (no validation split — validate on LB)
print(f'Training on {len(train_df)} rows')
print(f'Target distribution:')
for c in ['winner_model_a', 'winner_model_b', 'winner_tie']:
    print(f'  {c}: {train_df[c].mean():.3f}')

Competition train: 57477 rows
Training on 57477 rows
Target distribution:
  winner_model_a: 0.349
  winner_model_b: 0.342
  winner_tie: 0.309


In [4]:
class PreferenceDataset(Dataset):
    """Dataset for chatbot preference prediction."""
    
    def __init__(self, df, tokenizer, max_length, label_smoothing=0.0):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.label_smoothing = label_smoothing
        self.has_labels = 'winner_model_a' in df.columns
        # Left truncation: if text is too long, cut from the beginning
        # (end of responses is usually more informative than the start)
        self.tokenizer.truncation_side = 'left'
    
    def __len__(self):
        return len(self.df)
    
    def _format_conversation(self, row):
        """Format multi-turn conversation into a single string."""
        prompts = json.loads(row['prompt'])
        responses_a = json.loads(row['response_a'])
        responses_b = json.loads(row['response_b'])
        
        rounds = []
        for i in range(len(prompts)):
            r_a = responses_a[i] if i < len(responses_a) else ''
            r_b = responses_b[i] if i < len(responses_b) else ''
            rounds.append(
                f'<prompt>\n{prompts[i]}\n</prompt>\n'
                f'<response_a>\n{r_a}\n</response_a>\n'
                f'<response_b>\n{r_b}\n</response_b>'
            )
        
        return '\n'.join(rounds)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = self._format_conversation(row)
        
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        item = {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
        }
        
        if self.has_labels:
            labels = np.array([
                row['winner_model_a'],
                row['winner_model_b'],
                row['winner_tie']
            ], dtype=np.float32)
            if self.label_smoothing > 0:
                labels = labels * (1 - self.label_smoothing) + self.label_smoothing / 3
            item['labels'] = torch.tensor(labels)
        
        return item

## 2. Load Model & Tokenizer

In [ ]:
# Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=CFG.use_4bit,
    bnb_4bit_quant_type=CFG.bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=getattr(torch, CFG.bnb_4bit_compute_dtype),
    bnb_4bit_use_double_quant=CFG.use_double_quant,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(CFG.model_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
print(f'Tokenizer loaded: vocab_size={tokenizer.vocab_size}')

# Load model for sequence classification
# IMPORTANT: Disable soft-capping in config BEFORE loading so SDPA attention is used
from transformers import AutoConfig
model_config = AutoConfig.from_pretrained(CFG.model_path)
model_config.attn_logit_softcapping = None
model_config.final_logit_softcapping = None
model_config.num_labels = CFG.num_labels

model = AutoModelForSequenceClassification.from_pretrained(
    CFG.model_path,
    config=model_config,
    quantization_config=bnb_config,
    device_map=CFG.device_map,  # single GPU for 2B, 'auto' for 9B
    torch_dtype=torch.float16,
    attn_implementation='sdpa',
)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

print(f'Attention implementation: {model.config._attn_implementation}')
print(f'Model loaded on: {getattr(model, "hf_device_map", "single device")}')

In [ ]:
# Prepare for QLoRA training
model = prepare_model_for_kbit_training(model)

# Freeze early layers if configured
if CFG.freeze_layers > 0:
    for i, layer in enumerate(model.model.layers):
        if i < CFG.freeze_layers:
            for param in layer.parameters():
                param.requires_grad = False
    print(f'Froze first {CFG.freeze_layers} layers')

# LoRA config
lora_config = LoraConfig(
    r=CFG.lora_r,
    lora_alpha=CFG.lora_alpha,
    target_modules=CFG.lora_target_modules,
    lora_dropout=CFG.lora_dropout,
    bias='none',
    task_type=TaskType.SEQ_CLS,
    modules_to_save=['score'],  # keep classification head trainable
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Preserve device_map info after PEFT wrapping so Trainer knows model is already parallelized
# Without this, Trainer tries to wrap in DataParallel which conflicts with device_map='auto'
if hasattr(model, 'base_model') and hasattr(model.base_model, 'model'):
    base = model.base_model.model
    if hasattr(base, 'base_model') and hasattr(base.base_model, 'model'):
        inner = base.base_model.model
        if hasattr(inner, 'hf_device_map'):
            model.hf_device_map = inner.hf_device_map
            print(f'Preserved hf_device_map: {model.hf_device_map}')
    elif hasattr(base, 'hf_device_map'):
        model.hf_device_map = base.hf_device_map
        print(f'Preserved hf_device_map: {model.hf_device_map}')

# Re-initialize score head with small random weights (scaled by hidden size)
hidden_size = model_config.hidden_size
init_std = 0.02 / (hidden_size ** 0.5)
for name, param in model.named_parameters():
    if 'score' in name and 'weight' in name:
        nn.init.normal_(param, mean=0.0, std=init_std)
        print(f'Re-initialized {name} with std={init_std:.6f}')

# Ensure Trainer knows this model is already parallelized via device_map
if torch.cuda.device_count() > 1 and not hasattr(model, 'hf_device_map'):
    model.is_parallelizable = True
    model.model_parallel = True
    print('Set model_parallel=True to prevent DataParallel wrapping')

## 3. Custom Trainer with Soft-Label Loss

In [ ]:
class PreferenceTrainer(Trainer):
    """Custom trainer that uses soft cross-entropy for soft labels."""
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        labels = inputs.pop('labels')  # (batch, 3) soft labels
        outputs = model(**inputs)
        logits = outputs.logits  # (batch, 3)
        
        # Soft cross-entropy: -sum(y * log_softmax(logits))
        # Compute in float32 for numerical stability
        logits = logits.float()
        labels = labels.float()
        log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
        per_sample_loss = -(labels * log_probs).sum(dim=-1)  # (batch,)
        
        # When num_items_in_batch is provided (modern Trainer with grad accum),
        # return SUM so the Trainer can properly average across all micro-batches
        # in the effective batch. Otherwise the logged loss is wrong by a factor
        # of gradient_accumulation_steps.
        if num_items_in_batch is not None:
            loss = per_sample_loss.sum() / num_items_in_batch
        else:
            loss = per_sample_loss.mean()
        
        return (loss, outputs) if return_outputs else loss
    
    def _wrap_model(self, model, training=True, dataloader=None):
        # Skip DataParallel/DDP wrapping — model is already distributed via device_map
        return model


In [ ]:
# Create dataset
train_dataset = PreferenceDataset(
    train_df, tokenizer, CFG.max_length,
    label_smoothing=CFG.label_smoothing
)
print(f'Dataset: {len(train_dataset)} samples')

# Quick sanity check
sample = train_dataset[0]
print(f'input_ids shape: {sample["input_ids"].shape}')
print(f'labels: {sample["labels"]}')
print(f'Decoded (first 200 chars): {tokenizer.decode(sample["input_ids"][:50])}')

In [ ]:
# ============================================================
# DIAGNOSTIC: Verify gradient flow before training
# ============================================================
model.train()
sample = train_dataset[0]
input_device = next(model.parameters()).device
input_ids = sample['input_ids'].unsqueeze(0).to(input_device)
attention_mask = sample['attention_mask'].unsqueeze(0).to(input_device)
labels = sample['labels'].unsqueeze(0).to(input_device)

outputs = model(input_ids=input_ids, attention_mask=attention_mask)
logits = outputs.logits.float()
log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
loss = -(labels.float() * log_probs).sum(dim=-1).mean()

loss.backward()

print(f'Loss: {loss.item():.4f} (should be ~1.0-1.2, NOT ~8.8)')
print(f'Logits: {logits.detach().cpu().numpy()} (should NOT be all ~0)')
print(f'Probs: {torch.softmax(logits, dim=-1).detach().cpu().numpy()}')
print()

# Check gradient magnitudes
lora_grads, score_grads = [], []
for name, param in model.named_parameters():
    if param.requires_grad and param.grad is not None:
        grad_norm = param.grad.norm().item()
        if 'lora' in name:
            lora_grads.append((name.split('.')[-2] + '.' + name.split('.')[-1], grad_norm))
        elif 'score' in name:
            score_grads.append((name, grad_norm))

print(f'Score head gradients ({len(score_grads)} params):')
for name, norm in score_grads:
    print(f'  {name}: grad_norm={norm:.6f}')

print(f'\nLoRA gradients ({len(lora_grads)} params, showing first 6):')
for name, norm in sorted(lora_grads, key=lambda x: -x[1])[:6]:
    print(f'  {name}: grad_norm={norm:.6f}')

if not lora_grads and not score_grads:
    print('WARNING: No gradients found! Something is broken.')
elif all(n < 1e-10 for _, n in lora_grads + score_grads):
    print('WARNING: All gradients are ~0. Gradient flow is broken.')
else:
    print('\nGradients look healthy — model should learn.')

model.zero_grad()

## 4. Train

In [ ]:
training_args = TrainingArguments(
    output_dir=CFG.output_dir,
    max_steps=CFG.max_steps,       # cap steps — loss plateaus early
    per_device_train_batch_size=CFG.per_device_batch_size,
    gradient_accumulation_steps=CFG.gradient_accumulation_steps,
    learning_rate=CFG.learning_rate,
    warmup_steps=CFG.warmup_steps,
    weight_decay=CFG.weight_decay,
    lr_scheduler_type='cosine',    # decay LR after plateau to squeeze out improvement
    fp16=True,
    max_grad_norm=1.0,
    logging_steps=50,
    save_strategy='steps',
    save_steps=1500,          # save at the end
    save_total_limit=1,       # keep only the last checkpoint
    seed=CFG.seed,
    report_to='none',
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    dataloader_pin_memory=True,
    dataloader_num_workers=4,
    remove_unused_columns=False,
)

trainer = PreferenceTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)
# Auto-resume from checkpoint if one exists (e.g., after a crash)
resume_ckpt = None
if os.path.isdir(CFG.output_dir):
    checkpoints = [d for d in os.listdir(CFG.output_dir) if d.startswith('checkpoint-')]
    if checkpoints:
        resume_ckpt = os.path.join(CFG.output_dir, sorted(checkpoints, key=lambda x: int(x.split('-')[-1]))[-1])
        print(f'Resuming from checkpoint: {resume_ckpt}')
    else:
        print('No checkpoint found — starting fresh')
else:
    print('Starting training from scratch...')

trainer.train(resume_from_checkpoint=resume_ckpt)


In [ ]:
# Save the LoRA adapter
model.save_pretrained(CFG.output_dir)
tokenizer.save_pretrained(CFG.output_dir)
print(f'Adapter saved to {CFG.output_dir}')

# Show saved files
for f in os.listdir(CFG.output_dir):
    size = os.path.getsize(os.path.join(CFG.output_dir, f)) / 1e6
    print(f'  {f}: {size:.1f} MB')

## 5. Quick Validation (on a small holdout)


In [ ]:
# Quick smoke test on 20 training samples (just a sanity check, not real validation)
model.eval()
val_subset = train_df.head(20)
val_dataset = PreferenceDataset(val_subset, tokenizer, CFG.max_length)
input_device = next(model.parameters()).device

all_preds = []
with torch.no_grad():
    for i in range(len(val_dataset)):
        batch = val_dataset[i]
        input_ids = batch['input_ids'].unsqueeze(0).to(input_device)
        attention_mask = batch['attention_mask'].unsqueeze(0).to(input_device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits.float(), dim=-1).cpu().numpy()[0]
        all_preds.append(probs)

preds = np.array(all_preds)
targets = val_subset[['winner_model_a', 'winner_model_b', 'winner_tie']].values
ll = log_loss(targets, preds)
print(f'Train-subset log loss (20 samples): {ll:.4f}')
print('(This is on training data — real performance will be on LB)')


In [ ]:
## 6. Inference on Test Set + Submission

Run inference using the just-trained model (still in memory).  
Uses TTA (Test-Time Augmentation) by swapping response_a/response_b and averaging.

In [ ]:
# Inference config
INF_MAX_LENGTH = 1024  # longer context for inference but capped to save time
INF_BATCH_SIZE = 4     # 2B fits batch=4 at 1024 tokens easily; 9B use 1
USE_TTA = True

if CFG.model_size == '9b':
    INF_BATCH_SIZE = 1  # 9B is memory-constrained

# Load test data
test_df = pd.read_csv(f'{CFG.data_path}/test.csv')
print(f'Test: {len(test_df)} rows')
print(f'Columns: {list(test_df.columns)}')
print(f'Inference: max_length={INF_MAX_LENGTH}, batch_size={INF_BATCH_SIZE}, TTA={USE_TTA}')


In [ ]:
class InferenceDataset(Dataset):
    """Dataset for inference — supports optional TTA (response swap)."""
    
    def __init__(self, df, tokenizer, max_length, swap=False):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.swap = swap
        self.tokenizer.truncation_side = 'left'
    
    def __len__(self):
        return len(self.df)
    
    def _format_conversation(self, row):
        prompts = json.loads(row['prompt'])
        if self.swap:
            responses_a = json.loads(row['response_b'])
            responses_b = json.loads(row['response_a'])
        else:
            responses_a = json.loads(row['response_a'])
            responses_b = json.loads(row['response_b'])
        
        rounds = []
        for i in range(len(prompts)):
            r_a = responses_a[i] if i < len(responses_a) else ''
            r_b = responses_b[i] if i < len(responses_b) else ''
            rounds.append(
                f'<prompt>\n{prompts[i]}\n</prompt>\n'
                f'<response_a>\n{r_a}\n</response_a>\n'
                f'<response_b>\n{r_b}\n</response_b>'
            )
        return '\n'.join(rounds)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = self._format_conversation(row)
        encoding = self.tokenizer(
            text, max_length=self.max_length, padding='max_length',
            truncation=True, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
        }

In [ ]:
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

def predict(model, dataset, batch_size=1):
    """Run inference and return softmax probabilities."""
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    input_device = next(model.parameters()).device
    all_probs = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Predicting'):
            input_ids = batch['input_ids'].to(input_device)
            attention_mask = batch['attention_mask'].to(input_device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(outputs.logits.float(), dim=-1).cpu().numpy()
            all_probs.append(probs)
    
    return np.concatenate(all_probs, axis=0)

# Normal prediction
model.eval()
print('Running normal inference...')
dataset_normal = InferenceDataset(test_df, tokenizer, INF_MAX_LENGTH, swap=False)
probs_normal = predict(model, dataset_normal, INF_BATCH_SIZE)
print(f'Normal preds shape: {probs_normal.shape}')

if USE_TTA:
    print('Running TTA (swapped responses)...')
    dataset_swap = InferenceDataset(test_df, tokenizer, INF_MAX_LENGTH, swap=True)
    probs_swap = predict(model, dataset_swap, INF_BATCH_SIZE)
    # Swap columns 0,1 back (model_a ↔ model_b), keep tie
    probs_swap_corrected = probs_swap[:, [1, 0, 2]]
    final_probs = (probs_normal + probs_swap_corrected) / 2
    print(f'TTA averaged. Shape: {final_probs.shape}')
else:
    final_probs = probs_normal

row_sums = final_probs.sum(axis=1)
print(f'Row sum stats: min={row_sums.min():.4f}, max={row_sums.max():.4f}, mean={row_sums.mean():.4f}')

In [ ]:
submission = pd.DataFrame({
    'id': test_df['id'],
    'winner_model_a': final_probs[:, 0],
    'winner_model_b': final_probs[:, 1],
    'winner_tie': final_probs[:, 2],
})

# Clip extreme probabilities and renormalize
prob_cols = ['winner_model_a', 'winner_model_b', 'winner_tie']
for col in prob_cols:
    submission[col] = submission[col].clip(1e-7, 1 - 1e-7)
row_sums = submission[prob_cols].sum(axis=1)
for col in prob_cols:
    submission[col] = submission[col] / row_sums

submission.to_csv('submission.csv', index=False)
print(f'Submission saved: {submission.shape}')
print(submission.head(10))
print(f'\nPrediction stats:')
print(submission[prob_cols].describe())

In [ ]:
# Final validation
assert submission.shape[0] == len(test_df), 'Row count mismatch!'
assert set(submission['id']) == set(test_df['id']), 'ID mismatch!'
assert all(submission[prob_cols].sum(axis=1).between(0.999, 1.001)), 'Probs dont sum to 1!'
assert not submission.isnull().any().any(), 'NaN values found!'
print('All checks passed! Ready to submit.')